# 02 — Variáveis exógenas (seção 5.1 do v5)

**Lê:** `corte.json` (01) e o cache SGS em `dados_brutos/macro_cache/` (baixa da API do BCB só se faltar).  
**Grava:** `exogenas_semanais`, `macro_mensal`, `calendario_eventos`, `exogenas_listas.json`.  
**Feriados:** fonte definida em `FERIADOS_FONTE` no config (BrasilAPI com cache em `dados_brutos/brasilapi_feriados/`, ou cálculo pela Páscoa). Datas comerciais são sempre calculadas.

O cache fica fora da pasta da execução de propósito: o BCB revisa séries (ex.: IBC-Br), e reutilizar o mesmo arquivo garante que todas as execuções usam a mesma vintage. Registre a data do download no texto do TCC.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "02_exogenas"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import exogenas

corte = A.carregar_json("corte")
TEST_END = pd.Timestamp(corte["TEST_END"])
macro_long, macro_wide = exogenas.montar_macro(C.MACRO_SERIES, C.EXOGENOUS_API_START, C.EXOGENOUS_API_END,
                                               C.caminho("macro_cache"))
FONTE_FERIADOS = C.get("FERIADOS_FONTE", "calculado")
events = exogenas.calendario_eventos(range(C.PANEL_START.year - 1, TEST_END.year + 2), fonte_feriados=FONTE_FERIADOS,
                                     cache_dir=C.caminho("brasilapi_feriados"))
print(f"Feriados: fonte = {FONTE_FERIADOS}")
exog, listas = exogenas.exogenas_semanais(events, macro_wide, C.MACRO_SERIES, C.PANEL_START, TEST_END,
                                          C.MACRO_LAG_MONTHS)
print(f"Macro: {len(C.MACRO_SERIES)} séries, {len(macro_wide)} meses, atraso de {C.MACRO_LAG_MONTHS} meses")
print(f"Calendário: {len(listas['CALENDAR_FEATURES'])} atributos, {len(events)} datas | exógenas: {len(listas['EXOGENOUS_FEATURES'])}")
display(exog.head())

In [ ]:
A.salvar_df(exog, "exogenas_semanais")
A.salvar_df(macro_long, "macro_mensal")
A.salvar_df(events, "calendario_eventos")
cache_hash = {p.name: __import__("tcc_fluxo.io", fromlist=["x"]).sha256_arquivo(p)
              for p in sorted(C.caminho("macro_cache").glob("*.csv"))}
feriados_hash = ({p.name: __import__("tcc_fluxo.io", fromlist=["x"]).sha256_arquivo(p)
                  for p in sorted(C.caminho("brasilapi_feriados").glob("*.json"))} if FONTE_FERIADOS == "brasilapi" else {})
A.salvar_json({**listas, "fonte": C.MACRO_SOURCE, "fonte_feriados": FONTE_FERIADOS, "feriados_cache_sha256": feriados_hash, "series_sgs": C.MACRO_SERIES,
               "lag_macro_meses": C.MACRO_LAG_MONTHS, "cache_sha256": cache_hash,
               "n_meses_macro": len(macro_wide), "n_semanas_exogenas": len(exog), "n_eventos": len(events)},
              "exogenas_listas")
A.tabela(exog, "calendario_macro_semanal.csv")

In [ ]:
A.finalizar()